# 🚀 HUẾ SMART TOURISM - GPU BACKEND SERVER TRÊN GOOGLE COLAB
**Đề tài Khóa luận Tốt nghiệp:** Hệ thống Hỏi đáp và Gợi ý Lịch trình Du lịch Thông minh dựa trên Spatial GraphRAG & LLM
---
Notebook này cho phép bạn chạy toàn bộ **FastAPI Backend + Ollama (Qwen2.5:3b) + ChromaDB + Spatial Knowledge Graph** hoàn toàn miễn phí trên GPU **Nvidia T4** của Google Colab.

### ⚠️ BƯỚC 0: BẬT GPU TRƯỚC KHI CHẠY (BẮT BUỘC)
1. Trên thanh menu Colab, chọn **Runtime** (Thời gian chạy) -> **Change runtime type** (Thay đổi loại thời gian chạy).
2. Tại mục **Hardware accelerator** (Bộ tăng tốc phần cứng), chọn **T4 GPU**.
3. Bấm **Save** (Lưu).

### BƯỚC 1: KIỂM TRA GPU NVIDIA T4

In [ ]:
!nvidia-smi

### BƯỚC 2: KẾT NỐI GOOGLE DRIVE (ĐỂ LẤY DỮ LIỆU CHROMADB)
> **Lưu ý:** Thư mục `chroma_db` (khoảng 4.5GB) của bạn cần được đưa lên Google Drive (ví dụ trong thư mục `KhoaLuan_Data/chroma_db`). Lệnh dưới đây sẽ kết nối Drive để sử dụng trực tiếp mà không cần upload lại mỗi lần mở Colab.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
print("✅ Đã kết nối Google Drive thành công!")

### BƯỚC 3: CLONE MÃ NGUỒN KHÓA LUẬN VÀ LIÊN KẾT DỮ LIỆU

In [ ]:
# 1. Clone repository nếu chưa có
%cd /content
if not os.path.exists('/content/KhoaLuanTotNghiep'):
    !git clone https://github.com/minhquy017/KhoaLuanTotNghiep.git
else:
    %cd /content/KhoaLuanTotNghiep
    !git pull

%cd /content/KhoaLuanTotNghiep

# 2. Tạo liên kết (symlink) thư mục chroma_db từ Google Drive vào project
# Đường dẫn mẫu trên Drive: /content/drive/MyDrive/KhoaLuan_Data/chroma_db
drive_chroma_path = "/content/drive/MyDrive/KhoaLuan_Data/chroma_db"
local_chroma_path = "/content/KhoaLuanTotNghiep/data/processed/chroma_db"

if os.path.exists(drive_chroma_path):
    if os.path.exists(local_chroma_path) or os.path.islink(local_chroma_path):
        !rm -rf {local_chroma_path}
    !ln -s "{drive_chroma_path}" "{local_chroma_path}"
    print(f"✅ Đã liên kết ChromaDB từ Drive vào: {local_chroma_path}")
else:
    print(f"⚠️ Không tìm thấy ChromaDB tại '{drive_chroma_path}'. Hãy kiểm tra lại đường dẫn thư mục trên Google Drive của bạn!")

### BƯỚC 4: CÀI ĐẶT CÁC THƯ VIỆN PYTHON CẦN THIẾT

In [ ]:
!pip install -q fastapi uvicorn langgraph langchain-core langchain-ollama chromadb sentence-transformers networkx duckduckgo-search pydantic python-dotenv

### BƯỚC 5: CÀI ĐẶT OLLAMA VÀ TẢI MÔ HÌNH QWEN2.5 (3B) TRÊN GPU

In [ ]:
import subprocess
import time

# 1. Cài đặt Ollama cho Linux
print("📥 Đang cài đặt Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Khởi chạy tiến trình Ollama daemon ngầm
print("⚡ Đang khởi động Ollama daemon...")
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# 3. Tải mô hình Qwen2.5:3b vào GPU
print("🔥 Đang tải mô hình qwen2.5:3b (chỉ mất ~20s trên đường truyền Colab)...")
!ollama pull qwen2.5:3b

print("✅ Ollama và Qwen2.5:3b đã sẵn sàng trên GPU T4!")

### BƯỚC 6: CÀI ĐẶT CLOUDFLARE TUNNEL (ĐƯỜNG HẦM CÔNG KHAI HTTPS MIỄN PHÍ)

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
print("✅ Đã cài đặt Cloudflare Tunnel thành công!")

### BƯỚC 7: KHỞI CHẠY BACKEND FASTAPI & MỞ ĐƯỜNG LINK PUBLIC
> Chạy cell này để khởi động Server. Đường link HTTPS công khai sẽ được in ra ở cuối. Bạn chỉ việc copy link đó dán vào `frontend/src/api.js`!

In [ ]:
import subprocess
import time
import re

%cd /content/KhoaLuanTotNghiep

# 1. Khởi chạy FastAPI Backend trên cổng 8000
print("🚀 Đang khởi động FastAPI Backend (port 8000)...")
backend_log = open("/content/backend.log", "w")
server_process = subprocess.Popen(
    ["uvicorn", "src.app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=backend_log,
    stderr=subprocess.STDOUT
)
time.sleep(4)

# 2. Kích hoạt Cloudflare Tunnel trỏ tới port 8000
print("🌐 Đang thiết lập đường hầm Cloudflare Tunnel...")
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = tunnel_process.stdout.readline()
    if not line:
        continue
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print("\n" + "="*70)
if public_url:
    print(f"🎉 BACKEND SERVER ĐÃ CHẠY THÀNH CÔNG TRÊN GPU T4!")
    print(f"🔗 PUBLIC API URL: {public_url}")
    print(f"📚 API DOCS (SWAGGER): {public_url}/docs")
    print("="*70)
    print("\n👉 BƯỚC TIẾP THEO Ở MÁY CỦA BẠN:")
    print(f"Mở file `frontend/src/api.js`, đổi dòng số 3 thành:")
    print(f"   const API_BASE_URL = '{public_url}';")
    print("Và trải nghiệm ứng dụng Web bình thường, máy tính không còn tốn 1% GPU nào!")
else:
    print("⚠️ Không tìm thấy URL Cloudflare trong 30 giây. Vui lòng kiểm tra lại log!")

# Giữ tiến trình sống để Colab không dừng
try:
    while True:
        time.sleep(10)
except KeyboardInterrupt:
    print("🛑 Đã dừng server.")